Practice code to play with uproot and data stuff

## Set up for Data

In [ ]:
# Import necessary libraries to start

import uproot
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt

In [ ]:
file_path = "root://xcache//store/user/lpcmetx/SIDM/ULSignalSamples/2024/nanoaodYear2024.root"
file = uproot.open(file_path)

In [ ]:
file.keys()

In [ ]:
file.classnames()

In [ ]:
t = file["Events"]
t.show()

In [ ]:
file["Events"].all_members

In [ ]:
t["nMuon"].array()

In [ ]:
t["Muon_pt"].array()

In [ ]:
t["nMuon"].array()

In [ ]:
muons = t.arrays(
    ["Muon_pt", "Muon_eta", "Muon_phi", "Muon_mass", "Muon_charge"]
)
muons

In [ ]:
trigger = t["HLT_DoubleMu48NoFiltersNoVtx"].array(entry_stop=1000)

print(trigger)

In [ ]:
muons[trigger]

In [ ]:
muon_mask = (muons["Muon_pt"] >30)

print (muon_mask)

In [ ]:
muon_mask[0:20].to_list()

In [ ]:
# We can apply muon_mask to the events "muons"
muons[muon_mask]

In [ ]:
print(muons["Muon_mass"])

In [ ]:
print(muons["Muon_charge"])

### Using ak for awkward arrays

In [ ]:
>>> array = ak.Array([[[1.1, 2.2, 3.3],
...                    [],
...                    [4.4, 5.5],
...                    [6.6]
...                   ],
...                   [],
...                   [[7.7],
...                    [8.8, 9.9]]
...                   ])

In [ ]:
ak.num(array, axis = 1)
#this will show the number of elements in the axis 1

In [ ]:
ak.num(array, axis=2)

In [ ]:
ak.num(array, axis=0)
#the axis 0 will give a scalar that is the length of the array

the array we created is a doubly nested array, this means thereis a list of lists, the length of the array in the 0 axis means there are three lists in the array, as we saw in the axis=1 command, the first list has 4 lists in it, the second has 0 and the last has 2. The axis=2 gives the number of elements for for each list inside each list.

In [ ]:
array[ak.num(array) > 0, 0]

In [ ]:
array.mask[ak.num(array) > 0][:,0]

## Histograms

In [ ]:
t["nMuon"].array()
#in order to refer to an object in the tree you need to type it like this
#the .array() makes it show up as an array and the "nMuon" is an object within the "t" tree

In [ ]:
plt.hist(t["nMuon"].array())
#this plots a histogram of the nMuon array

In [ ]:
Muon_pt = t["Muon_pt"].array()

In [ ]:
Flat = ak.flatten(Muon_pt, axis=1)

In [ ]:
Flat_pt = ak.flatten(Muon_pt, axis=None)

In [ ]:
print(Flat_pt)

In [ ]:
plt.hist(Flat_pt)

In [ ]:
import boost_histogram as bh

In [ ]:
import hist

h = hist.Hist(hist.axis.Regular(20, 0, 20, name="number of muons"))

h.fill(t["nMuon"].array())

h.plot();

## Masks Investigation

In [ ]:
# Let's start by looking at one of the masks from the Events TTree in the data file
# I just picked a random boolean array that started with HLT

print(t["HLT_EphemeralPhysics"].array())

In [ ]:
#Since the mask is longer than our data set (muons has 1000 entries) we need to slice the mask to match

mask1 = t["HLT_EphemeralPhysics"].array()[:1000]

In [ ]:
print(mask1)

In [ ]:
#Now we can create an array where we apply the mask to the "muons" that we set as a variable earlier 

mask1_data = muons[mask1]

In [ ]:
print(mask1_data)

In [ ]:
mask2 = t["HLT_DoubleL2Mu50"].array()[:1000]

In [ ]:
print(mask2)

In [ ]:
mask2_data = muons[mask2]

print(mask2_data)

In [ ]:
# To make a plot we cannot directly use plt.hist on the mask2_data because it is a jagged array

# First we will select the specific numerical field we want to plot (e.g., 'Muon_pt')
muon_pt_jagged = mask2_data["Muon_pt"]

# Then we can flatten the jagged data into a single 1D array of all muon pT values using the ak.flatten function
flat_muon_pt = ak.flatten(muon_pt_jagged, axis=None)

# Then plot the flattened data
plt.hist(flat_muon_pt, bins=100, range=(0,500))
plt.title('Histogram of Muon $p_T$ for Masked Events')
plt.xlabel('Muon $p_T$')
plt.ylabel('Count')
plt.show()

In [ ]:
#This would be more useful with both histograms plotted together of with and without the mask

# Muon_pt without the mask
plt.hist(Flat_pt, bins=100, range=(0,500))
plt.show()

# Muon_pt with the mask
plt.hist(flat_muon_pt, bins=100, range=(0,500))
plt.show()


In [ ]:
# This would be even better if we could get both in the same graph

plt.hist(x=[Flat_pt, flat_muon_pt], alpha=0.6, bins=100, range=(0,500))

Let's apply more than one mask to the muons array

In [ ]:
# We're going to create and apply two makss that pick out all muons with over 30GeV pt and less than 2.4 eta

# We already defined Muon_pt but we should also define Muon_eta

Muon_eta = t["Muon_eta"].array()

# Now to create the masks

pt_condition = Muon_pt > 30
eta_condition = Muon_eta < 2.4 

combined_muon_mask = pt_condition & eta_condition

#Since we are applying the masks to an awkward array not just a simple set of numbers, we need to use ak.any()

event_mask = ak.any(combined_muon_mask, axis=1)

# Finally we can apply the event_mask and we get the filtered events

filtered_events = muons[event_mask]

print(filtered_events.show())


In [ ]:
# Let's seee if we can do this again but with different conditions that are double-ended

pt_lower_bound = Muon_pt > 30.0
pt_upper_bound = Muon_pt < 1000.0
pt_condition2 = pt_lower_bound & pt_upper_bound

eta_lower_bound = Muon_eta > -2.4
eta_upper_bound = Muon_eta < 2.4
eta_condition2 = eta_lower_bound & eta_upper_bound

combined_muon_mask2 = pt_condition & eta_condition

event_mask2 = ak.any(combined_muon_mask2, axis=1)

filtered_events2 = muons[event_mask2]

print(filtered_events2.show())



### For Loop to check triggers

In [ ]:
# Now we're going to create a for loop that checks how many events pass each trigger 

# Recall that we have "muons" to refer to all the muon events, and Muon_pt and Muon_eta defined already
# First we need to isolate the names of the triggers we want to analyze

# Let's just start with the original file

file_path = "root://xcache//store/user/lpcmetx/SIDM/ULSignalSamples/2024/nanoaodYear2024.root"
file = uproot.open(file_path)

#We'll define the Events object in the file

Events = file["Events"]

# then we want to use the .keys() to create a list of all the branch names in "Events" 
# We showed all the names and their attributes earlier in the code but here we want just a simple list of the names only so its easy to see

all_branch_names = list(Events.keys())

# Check that it worked using the print function
print(all_branch_names)


In [ ]:
# Now we can make a simple for loop to print only the first dozen items that start with "HLT_" 
# this gives us a list of names we can use when we want to analyze triggers later

hlt_triggers = []
for name in all_branch_names:
    if name.startswith("HLT_"):
        hlt_triggers.append(name)

print(hlt_triggers[:20])

# This is mostly so I don't have to keep scrolling back up to the beginning of the notebook and then scroll through the list of 
# items in events to find the HLT stuff. I'm slicing it at 20 but if I wanted to see more or different ones we could change the slicing
# i.e. 20:40 to see the next 20 triggers that start with HLT

In [ ]:
# We now have a list of triggers we want to analyze, lets make a for loop to analyze them

N_EVENTS = Events.num_entries # Get the total number of events

# To make it easy to refer to different triggers we can make a triggers dictionary, all we have to do is set the name, 
# then write the code that refers back to the branch from the Events TTree. To add more triggers to the dictionary just copy and
# paste the format of a previous one and change the name. We have a list of names from the previous cell.

triggers = {
    "HLT_EphemeralPhysics": Events["HLT_EphemeralPhysics"].array(),
    "HLT_EphemeralZeroBias": Events["HLT_EphemeralZeroBias"].array(),
    "HLT_EcalCalibration": Events["HLT_EcalCalibration"].array(),
    "HLT_HcalCalibration": Events["HLT_HcalCalibration"].array()
}
    
# Analyze Triggers (This is the main loop where we'll get the number of events passed for each trigger)

# first we'll define the array of results, we'll add entries to this as we apply the masks and get pass numbers

analysis_results = []

for name, mask_array in triggers.items():
    # The .items() function allows us to use the dictionary we set up so that each iteration of the for loop we can look at both
    # the name and the value, "for name" references the "HLT_..." part of the dictionary that is the names of the triggers. The
    # "mask_array" part refers to the dictionary value which is the actual data mask array
   
    passed_count = np.sum(mask_array)
    # the function numpy.sum() treats True as 1 and False as 0 
    # this means for passed_count, np.sum simply adds up all the 1s (the True values), giving you the total events that passed
    
    # Calculate the percentage of events that pass the trigger 
    # this is an easy math function, we don't necessarily need it but it could be useful to see
    pass_percentage = (passed_count / N_EVENTS) * 100
    
    # Store the results by appending them to the analysis results array
    # In the array we want to store multiple things for each item (like how in "muons" each muon has different variables stored)
    # To do this we use another dictionary where there is a name, count, and percent, for which we store different values
    analysis_results.append({
        "name": name,
        "count": passed_count,
        "percent": pass_percentage
    })

# Sort and Display Results

# Sort the results in descending order based on 'count'
# To do this we can use the pyhton sorted() function to create a new sorted list that doesn't modify the original list
# the key part tells python what to sort it by, we use lambda x: x["count"] to tell it to look at the count value for sorting
# reverse=True makes the new list sorted from largest values to smallest

sorted_results = sorted(analysis_results, key=lambda x: x["count"], reverse=True)

# To make the output easy to read we can add in a title and make it into a table

print(f"TRIGGER ANALYSIS REPORT (Total Events = {N_EVENTS})")  # title line
print("-" * 60)                                            # separate the title with a bunch of dashes

print(f"{'Trigger Name':<30} | {'Events Passed':>15} | {'Pass Rate':>10}")  # make 3 columns for the trigger, number events, and pass rate
print("-" * 60)

# then we can print the results using another for loop
# in this one we just need to say that for each item in the "sorted_results" array we want to print the name string and the count and percent numbers

for result in sorted_results:
    print(
        f"{result['name']:<30} | "
        f"{result['count']:>15,} | "  # Use comma for thousands separator
        f"{result['percent']:>8.2f}%"
    )


In [ ]:
#Since it looks like a lot of the initial ones are just 100 percent pass rate, I'm going to try to 
# get a list of all the HLT_Mu triggers because they are more likely to have varying pass rates.

mu_hlt_triggers = []
for name in all_branch_names:
    if name.startswith("HLT_Mu"):
        mu_hlt_triggers.append(name)

print(mu_hlt_triggers[:20])

In [ ]:
# Now lets run the same code with the HLT_Mu triggers

N_EVENTS = Events.num_entries # Get the total number of events

triggers = {
    "HLT_Mu27_Ele37_CaloIdL_MW": Events["HLT_Mu27_Ele37_CaloIdL_MW"].array(),
    "HLT_Mu37_Ele27_CaloIdL_MW": Events["HLT_Mu37_Ele27_CaloIdL_MW"].array(),
    "HLT_Mu37_TkMu27": Events["HLT_Mu37_TkMu27"].array(),
    "HLT_Mu0_L1DoubleMu": Events["HLT_Mu0_L1DoubleMu"].array(),
    "HLT_Mu4_L1DoubleMu": Events["HLT_Mu4_L1DoubleMu"].array(),
    "HLT_Mu3_PFJet40": Events["HLT_Mu3_PFJet40"].array(),
    "HLT_Mu7p5_L2Mu2_Jpsi": Events["HLT_Mu7p5_L2Mu2_Jpsi"].array(),
    "HLT_Mu7p5_L2Mu2_Upsilon": Events["HLT_Mu7p5_L2Mu2_Upsilon"].array(),
    "HLT_Mu3_L1SingleMu5orSingleMu7": Events["HLT_Mu3_L1SingleMu5orSingleMu7"].array(),
    "HLT_Mu0_Barrel": Events["HLT_Mu0_Barrel"].array(),
    "HLT_Mu0_Barrel_L1HP6": Events["HLT_Mu0_Barrel_L1HP6"].array(),
    "HLT_Mu0_Barrel_L1HP7": Events["HLT_Mu0_Barrel_L1HP7"].array()
}

# There's probably a better way to reference triggers and their names without having to type them all out but I haven't
# figured that out yet. Maybe another for loop to create an array that makes the dictionary for us?

# Analyze Triggers 

analysis_results = []

for name, mask_array in triggers.items():

    # Count the number of events that pass 
    passed_count = np.sum(mask_array)
    
    # Calculate the percentage of events that pass the trigger 
    pass_percentage = (passed_count / N_EVENTS) * 100
    
    # Store the results by appending them to the analysis results array
    analysis_results.append({
        "name": name,
        "count": passed_count,
        "percent": pass_percentage
    })

# Sort and Display Results

sorted_results = sorted(analysis_results, key=lambda x: x["count"], reverse=True)

# Make table

print(f"TRIGGER ANALYSIS REPORT (Total Events = {N_EVENTS})")  # title line
print("-" * 60)                                            # separate the title with a bunch of dashes

print(f"{'Trigger Name':<30} | {'Events Passed':>15} | {'Pass Rate':>10}")  # make 3 columns 
print("-" * 60)

# Print the results
for result in sorted_results:
    print(
        f"{result['name']:<30} | "
        f"{result['count']:>15,} | "  
        f"{result['percent']:>8.2f}%"
    )


In [ ]:
# Just to double check that this is correct I'm going to separately apply one of the masks to muons and see if
# it returns the right number of muon events. The smallest count is 4 so I'll use that one since it should be easy to check

HLT_Mu27_mask = Events["HLT_Mu27_Ele37_CaloIdL_MW"].array()

mask_applied = muons[HLT_Mu27_mask]

# We applied the mask, now we just need to get the number of muons from the result
# For this use the ak.num() function and set the axis to 0 since we want the total muons and don't really care about the pt and eta numbers 

ak.num(mask_applied, axis=0)

The ak.num confirmed that the mask gives us 4 events that pass which means our code worked!

## Git Stuff

In [ ]:
# Since this is in a notebook and git commands are for shell we need to add an exclamation mark at the beginning to tell
# Python to pass the command directly to the shell

!git add Practice_Notebook.ipynb

!git status

In [ ]:
!git commit -m "Most recent version of practice notebook"

!git status